In [1]:
from pathlib import Path
import numpy as np,pandas as pd

ROOT=Path("..")
C=ROOT/"results"/"week12"/"confirmatory"
A=C/"analysis"
M=A/"formal_model"
F=A/"failure_modes"
B=ROOT/"results"/"week13"/"bge_confirmatory"/"analysis"

OUT=ROOT/"results"/"week12"/"final_reporting"
OUT.mkdir(parents=True,exist_ok=True)

In [2]:
# load reporting sources 
model=pd.read_csv(M/"primary_model_dataset.csv")
fm=pd.read_csv(F/"failure_mode_table.csv")
type1=pd.read_csv(A/"type1_wilson.csv")

print("Model cells:",len(model))
print("Failure-mode cells:",len(fm))
print("Formal null cells:",len(type1))

Model cells: 185
Failure-mode cells: 185
Formal null cells: 40


In [3]:
# Build historical N=100 null table
TESTS=["C2ST-Linear","C2ST-RBF","Energy","MMD","Sinkhorn"]

hist={
"Broken Timing Structural":[.035,.040,.065,.055,.045],
"Campaign MiniLM":[.035,.025,.055,.055,.065],
"Message MiniLM":[.050,.045,.045,.050,.050],
"Template TF-IDF":[.030,.025,.045,.045,.045]
}

rows=[]
for r,v in hist.items():
    for t,p in zip(TESTS,v):
        rows.append({
            "representation":r,"N":100,"test":t,
            "type_I_error":p,"rejections":round(p*200),"n_reps":200
        })

null100=pd.DataFrame(rows)

def wilson(k,n,z=1.959963984540054):
    p=k/n; z2=z*z; d=1+z2/n
    c=(p+z2/(2*n))/d
    h=z*np.sqrt(p*(1-p)/n+z2/(4*n*n))/d
    return c-h,c+h

ci=null100.apply(lambda x:wilson(x.rejections,x.n_reps),axis=1)
null100["type1_wilson_low"]=[x[0] for x in ci]
null100["type1_wilson_high"]=[x[1] for x in ci]
null100["calibration_status"]=np.where(
    null100.rejections.between(4,16),
    "within preregistered range","FLAG"
)

null100.head()

,representation,N,test,type_I_error,rejections,n_reps,type1_wilson_low,type1_wilson_high,calibration_status
0,Broken Timing Structural,100,C2ST-Linear,0.035,7,200,0.017056,0.070471,within preregistered range
1,Broken Timing Structural,100,C2ST-RBF,0.040,8,200,0.020406,0.076932,within preregistered range
2,Broken Timing Structural,100,Energy,0.065,13,200,0.038376,0.108019,within preregistered range
3,Broken Timing Structural,100,MMD,0.055,11,200,0.030985,0.095787,within preregistered range
4,Broken Timing Structural,100,Sinkhorn,0.045,9,200,0.023853,0.083297,within preregistered range


In [4]:
# Normalize formal N=50/200 null table
null_new=type1.rename(columns={
    "wilson_low":"type1_wilson_low",
    "wilson_high":"type1_wilson_high"
})[
    ["representation","N","test","type_I_error",
     "type1_wilson_low","type1_wilson_high","calibration_status"]
]

null100=null100[
    ["representation","N","test","type_I_error",
     "type1_wilson_low","type1_wilson_high","calibration_status"]
]

null_all=pd.concat([null_new,null100],ignore_index=True)

assert len(null_all)==60


In [5]:
# Add representation + power Wilson intervals
REP={
"Broken Timing":"Broken Timing Structural",
"Entity Inconsistency":"Campaign MiniLM",
"Missing Cross-Channel Reference":"Campaign MiniLM",
"Over-Polished Language":"Message MiniLM",
"Template Repetition":"Template TF-IDF"
}

master=model.copy()
master["representation"]=master.scenario.map(REP)

ci=master.apply(lambda x:wilson(x.rejections,x.n_reps),axis=1)
master["power_wilson_low"]=[x[0] for x in ci]
master["power_wilson_high"]=[x[1] for x in ci]

master.head()

,scenario,severity,N,test,rejections,n_reps,power,source,representation,power_wilson_low,power_wilson_high
0,Broken Timing,0.1,50,Energy,77,200,0.385,confirmatory_n50_n200,Broken Timing Structural,0.320333,0.454001
1,Broken Timing,0.1,50,MMD,75,200,0.375,confirmatory_n50_n200,Broken Timing Structural,0.310854,0.443857
2,Broken Timing,0.1,50,Sinkhorn,10,200,0.050,confirmatory_n50_n200,Broken Timing Structural,0.027383,0.089578
3,Broken Timing,0.1,200,C2ST-Linear,67,200,0.335,confirmatory_n50_n200,Broken Timing Structural,0.273241,0.402978
4,Broken Timing,0.1,200,C2ST-RBF,67,200,0.335,confirmatory_n50_n200,Broken Timing Structural,0.273241,0.402978


In [6]:
# Attach matched null calibration
master=master.merge(
    null_all,
    on=["representation","N","test"],
    how="left",
    validate="many_to_one"
)

assert len(master)==185
assert master.type_I_error.notna().all()

In [7]:
# Attach failure modes
fm2=fm[
    ["scenario","severity","N","test","failure_mode"]
].copy()

master=master.merge(
    fm2,
    on=["scenario","severity","N","test"],
    how="left",
    validate="one_to_one"
)

assert len(master)==185
assert master.failure_mode.notna().all()

In [8]:
# final master table 
cols=[
"scenario","representation","severity","N","test","source",
"rejections","n_reps","power",
"power_wilson_low","power_wilson_high",
"type_I_error","type1_wilson_low","type1_wilson_high",
"calibration_status","failure_mode"
]

master=master[cols].sort_values(
    ["scenario","severity","N","test"]
).reset_index(drop=True)

master.to_csv(OUT/"primary_master_reporting_table.csv",index=False)

print("Rows:",len(master))
master.head()

Rows: 185


,scenario,representation,severity,N,test,source,rejections,n_reps,power,power_wilson_low,power_wilson_high,type_I_error,type1_wilson_low,type1_wilson_high,calibration_status,failure_mode
0,Broken Timing,Broken Timing Structural,0.1,50,Energy,confirmatory_n50_n200,77,200,0.385,0.320333,0.454001,0.050,0.027383,0.089578,within preregistered range,low
1,Broken Timing,Broken Timing Structural,0.1,50,MMD,confirmatory_n50_n200,75,200,0.375,0.310854,0.443857,0.055,0.030985,0.095787,within preregistered range,low
2,Broken Timing,Broken Timing Structural,0.1,50,Sinkhorn,confirmatory_n50_n200,10,200,0.050,0.027383,0.089578,0.035,0.017056,0.070471,within preregistered range,near-null
3,Broken Timing,Broken Timing Structural,0.1,100,C2ST-Linear,historical_n100_anchor,5,200,0.025,0.010725,0.057178,0.035,0.017056,0.070471,within preregistered range,near-null
4,Broken Timing,Broken Timing Structural,0.1,100,C2ST-RBF,historical_n100_anchor,8,200,0.040,0.020406,0.076932,0.040,0.020406,0.076932,within preregistered range,near-null


In [9]:
# final master integrity 
assert len(master)==185
assert master.duplicated(["scenario","severity","N","test"]).sum()==0
assert master[cols].notna().all().all()
assert set(master.source)=={"historical_n100_anchor","confirmatory_n50_n200"}

print("FINAL PRIMARY REPORTING TABLE PASSED")

FINAL PRIMARY REPORTING TABLE PASSED


### Runtime summary 


In [10]:
# Load raw formal runtimes
power=pd.read_csv(C/"primary_power_runs.csv")
null=pd.read_csv(C/"formal_clean_null_runs.csv")

runtime_raw=pd.concat([
    power.assign(run_type="power"),
    null.assign(run_type="null")
],ignore_index=True)

runtime_raw.shape

(30000, 19)

In [11]:
# Primary runtime summary by test and N
runtime=(
    runtime_raw.groupby(["run_type","N","test"],as_index=False)
    .agg(
        runs=("runtime","size"),
        mean_runtime=("runtime","mean"),
        median_runtime=("runtime","median"),
        total_runtime=("runtime","sum")
    )
)

runtime.to_csv(OUT/"primary_runtime_summary.csv",index=False)
runtime.round(4)

,run_type,N,test,runs,mean_runtime,median_runtime,total_runtime
0,null,50,C2ST-Linear,800,0.0179,0.0129,14.3202
1,null,50,C2ST-RBF,800,0.0149,0.0094,11.9392
2,null,50,Energy,800,0.0208,0.0192,16.6203
3,null,50,MMD,800,0.0206,0.0197,16.4616
4,null,50,Sinkhorn,800,0.9470,0.7732,757.6132
5,null,200,C2ST-Linear,800,0.0254,0.0158,20.3383
6,null,200,C2ST-RBF,800,0.1114,0.0275,89.0958
7,null,200,Energy,800,0.1192,0.1046,95.3378
8,null,200,MMD,800,0.1172,0.1020,93.7633
9,null,200,Sinkhorn,800,2.1067,1.2251,1685.3481


In [12]:
# power only runtime table 
runtime_power=(
    power.groupby(["N","test"],as_index=False)
    .agg(
        runs=("runtime","size"),
        mean_runtime=("runtime","mean"),
        median_runtime=("runtime","median"),
        total_runtime=("runtime","sum")
    )
)

runtime_power.round(4)

,N,test,runs,mean_runtime,median_runtime,total_runtime
0,50,C2ST-Linear,2000,0.0143,0.0124,28.6930
1,50,C2ST-RBF,2000,0.0121,0.0076,24.1361
2,50,Energy,3000,0.0193,0.0161,57.8695
3,50,MMD,3000,0.0207,0.0163,62.1614
4,50,Sinkhorn,2000,0.8592,0.6791,1718.3246
5,200,C2ST-Linear,2400,0.0143,0.0108,34.3433
6,200,C2ST-RBF,2400,0.0504,0.0221,121.0702
7,200,Energy,1400,0.1152,0.0825,161.3189
8,200,MMD,1400,0.1195,0.0840,167.3402
9,200,Sinkhorn,2400,1.4648,1.0640,3515.5741


###  BGE reporting table 

In [13]:
# load bge 
bge=pd.read_csv(B/"bge_power_with_calibration.csv")

print("BGE cells:",len(bge))
assert len(bge)==30

BGE cells: 30


In [14]:
# compact bge reporting table 
bge_final=bge[
    ["scenario","severity","test","rejections","n_reps","power",
     "wilson_low","wilson_high","type_I_error",
     "type1_wilson_low","type1_wilson_high","calibration_status"]
].rename(columns={
    "wilson_low":"power_wilson_low",
    "wilson_high":"power_wilson_high"
})

bge_final["representation"]="BGE Campaign"
bge_final["N"]=100
bge_final["source"]="secondary_bge_robustness"

bge_final.to_csv(OUT/"bge_master_reporting_table.csv",index=False)

bge_final.head()

,scenario,severity,test,rejections,n_reps,power,power_wilson_low,power_wilson_high,type_I_error,type1_wilson_low,type1_wilson_high,calibration_status,representation,N,source
0,Entity Inconsistency,0.1,C2ST-Linear,1,200,0.005,8.831687e-04,0.027774,0.060,0.034652,0.101932,within preregistered range,BGE Campaign,100,secondary_bge_robustness
1,Entity Inconsistency,0.1,C2ST-RBF,0,200,0.000,1.734723e-18,0.018845,0.050,0.027383,0.089578,within preregistered range,BGE Campaign,100,secondary_bge_robustness
2,Entity Inconsistency,0.1,Energy,171,200,0.855,7.995133e-01,0.897106,0.045,0.023853,0.083297,within preregistered range,BGE Campaign,100,secondary_bge_robustness
3,Entity Inconsistency,0.1,MMD,170,200,0.850,7.939442e-01,0.892864,0.040,0.020406,0.076932,within preregistered range,BGE Campaign,100,secondary_bge_robustness
4,Entity Inconsistency,0.1,Sinkhorn,2,200,0.010,2.746658e-03,0.035722,0.035,0.017056,0.070471,within preregistered range,BGE Campaign,100,secondary_bge_robustness


In [15]:
# load MiniLM vs BGE comparison
comparison=pd.read_csv(B/"bge_vs_minilm_n100.csv")
comparison.to_csv(OUT/"bge_vs_minilm_n100.csv",index=False)

comparison.round(3)

,scenario,severity,test,bge_power,minilm_power,delta_bge_minus_minilm
0,Entity Inconsistency,0.1,C2ST-Linear,0.005,0.000,0.005
1,Entity Inconsistency,0.1,C2ST-RBF,0.000,0.000,0.000
2,Entity Inconsistency,0.1,Energy,0.855,0.605,0.250
3,Entity Inconsistency,0.1,MMD,0.850,0.640,0.210
4,Entity Inconsistency,0.1,Sinkhorn,0.010,0.045,-0.035
5,Entity Inconsistency,0.3,C2ST-Linear,0.290,0.025,0.265
6,Entity Inconsistency,0.3,C2ST-RBF,0.255,0.015,0.240
7,Entity Inconsistency,0.3,Energy,1.000,1.000,0.000
8,Entity Inconsistency,0.3,MMD,1.000,1.000,0.000
9,Entity Inconsistency,0.3,Sinkhorn,0.020,0.025,-0.005


### Final reporting summaries 

In [16]:
# calibration summary 
cal_summary=pd.DataFrame({
    "set":["Primary N=50/200","Historical N=100","BGE"],
    "within_range":[
        (null_new.calibration_status!="FLAG").sum(),
        (null100.calibration_status!="FLAG").sum(),
        5
    ],
    "flagged":[
        (null_new.calibration_status=="FLAG").sum(),
        (null100.calibration_status=="FLAG").sum(),
        0
    ]
})

cal_summary

,set,within_range,flagged
0,Primary N=50/200,36,4
1,Historical N=100,20,0
2,BGE,5,0


In [17]:
# failure mode summary
fm_summary=(
    master.groupby(["source","test","failure_mode"])
    .size()
    .rename("cells")
    .reset_index()
)

fm_summary.to_csv(OUT/"failure_mode_summary.csv",index=False)
fm_summary

,source,test,failure_mode,cells
0,confirmatory_n50_n200,C2ST-Linear,high,5
1,confirmatory_n50_n200,C2ST-Linear,low,6
2,confirmatory_n50_n200,C2ST-Linear,near-null,7
3,confirmatory_n50_n200,C2ST-Linear,saturated,4
4,confirmatory_n50_n200,C2ST-RBF,high,3
5,confirmatory_n50_n200,C2ST-RBF,low,6
6,confirmatory_n50_n200,C2ST-RBF,moderate,2
7,confirmatory_n50_n200,C2ST-RBF,near-null,7
8,confirmatory_n50_n200,C2ST-RBF,saturated,4
9,confirmatory_n50_n200,Energy,high,1


In [18]:
# key inferential results
lrt=pd.read_csv(M/"primary_interaction_LRT.csv")

lrt["p_report"]=np.where(
    lrt.p_value<.001,"<.001",lrt.p_value.round(4).astype(str)
)

lrt

,interaction,LR,df,p_value,p_report
0,Scenario × Test,5014.185529,16,0.000000e+00,<.001
1,Severity × Test,1046.823757,8,1.164074e-220,<.001
2,N × Test,173.295525,8,2.627829e-33,<.001


## Methodological Incidents and Deviations

This section records operational issues, methodological amendments, and calibration diagnostics observed during the confirmatory phase. No post hoc retuning of statistical tests, significance thresholds, permutation counts, representation parameters, pruning rules, or frozen metric parameters was performed after inspection of confirmatory outcomes.

### 1. Formal clean-null sampling at N=200

**Original issue.**  
The evaluation pool contained 300 units. Therefore, two disjoint clean groups of N=200 could not be sampled from this pool without overlap.

**Resolution.**  
Formal clean-null calibration used the frozen 700-unit clean reference representation. For each Monte Carlo repetition, 2N distinct clean units were sampled without replacement and split into P and Q. At N=200, this produced two disjoint groups of 200 units.

**Impact on scientific design.**  
The modification affected only formal clean-null calibration. Defect-power comparisons continued to use the frozen 300-unit evaluation pool and the preregistered paired clean-versus-defective sampling design.

### 2. Pandas parsing of the manifest `kind="null"` field

**Observed issue.**  
During the final smoke check, `pandas.read_csv()` interpreted the literal string `"null"` as a missing value under its default NA parsing behavior.

**Resolution.**  
Manifest and sampling-plan reads were changed to use `keep_default_na=False` with explicit empty-string NA handling.

**Impact on scientific design.**  
None. The frozen manifest, sampling plan, seeds, samples, and hashes were not modified. This was a read-side operational correction only.


### 3. Conservative primary clean-null configurations

Four of the 40 formal primary clean-null configurations fell outside the preregistered 4–16 rejection calibration range:

- Broken Timing Structural, N=50, C2ST-RBF: 3/200 rejections.
- Message MiniLM, N=50, C2ST-RBF: 1/200.
- Template TF-IDF, N=200, C2ST-Linear: 2/200.
- Template TF-IDF, N=200, C2ST-RBF: 2/200.

All four deviations were conservative rather than anti-conservative.

**Action taken.**  
No retuning or exclusion was performed. Corresponding power results were retained and interpreted with calibration caution.

### 4. Secondary BGE robustness analysis

The BGE analysis was prespecified as a secondary encoder-robustness experiment restricted to Entity Inconsistency and Missing Cross-Channel Reference at N=100.

It used the frozen BGE representation-specific parameters:

- MMD sigma = 0.3873605130049743
- Sinkhorn blur = 0.038736051300497475

All five BGE clean-null calibration conditions fell within the preregistered calibration range.

**Impact on primary analysis.**  
None. BGE results were not used to redefine primary hypotheses, pruning decisions, metric settings, or primary conclusions.

In [19]:
# final inventory 

print("FINAL REPORTING OUTPUTS")
for p in sorted(OUT.iterdir()):
    print("-",p.name)

FINAL REPORTING OUTPUTS
- bge_master_reporting_table.csv
- bge_vs_minilm_n100.csv
- failure_mode_summary.csv
- primary_master_reporting_table.csv
- primary_runtime_summary.csv
